# GREEN_CIRCLE Notebook 1 — Entraînement DeepLabV3
**ISIC Project | Segmentation des Lésions Cutanées**

Ce notebook entraîne uniquement le modèle DeepLabV3 sur le dataset ISIC 2018.

## SETTING Étape 1 — Setup & Création des dossiers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

PROJECT_PATH = '/content/drive/MyDrive/ISIC_Project'
IMAGES_PATH  = os.path.join(PROJECT_PATH, 'data', 'Images')
MASQUES_PATH = os.path.join(PROJECT_PATH, 'data', 'Masques')
OUTPUTS_PATH = os.path.join(PROJECT_PATH, 'outputs')

# ── Dossier pour DeepLabV3 ────────────────────────────────────────────────────────
DIRS = {
    'deeplabv3' : os.path.join(OUTPUTS_PATH, 'deeplabv3'),
}
for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)

sys.path.append(os.path.join(PROJECT_PATH, 'src'))

print('OK Structure de dossiers créée :')
for name, path in DIRS.items():
    print(f'   FOLDER outputs/{name}/')

## PACKAGE Étape 2 — Installations & Imports

In [ ]:
!pip install -q albumentations opencv-python-headless scikit-learn seaborn torchvision

import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models.segmentation as seg_models
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
from tqdm import tqdm
import json, time

## CARD_INDEX Étape 3 — Dataset & DataLoaders

In [ ]:
IMG_SIZE   = 256
BATCH_SIZE = 8
EPOCHS     = 30
THRESHOLD  = 0.5

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3), A.ColorJitter(p=0.4), A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])
val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

class ISICDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img  = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']; mask = out['mask'].unsqueeze(0)
        return img.float(), mask.float()

IMAGE_EXTENSIONS = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
all_images = sorted([p for p in Path(IMAGES_PATH).glob('*.*') if p.suffix.lower() in IMAGE_EXTENSIONS])
all_masks  = sorted([p for p in Path(MASQUES_PATH).glob('*.*') if p.suffix.lower() in IMAGE_EXTENSIONS])
assert len(all_images) == len(all_masks)

train_imgs, temp_imgs, train_msks, temp_msks = train_test_split(
    all_images, all_masks, test_size=0.3, random_state=42)
val_imgs, test_imgs, val_msks, test_msks = train_test_split(
    temp_imgs, temp_msks, test_size=0.5, random_state=42)

train_loader = DataLoader(ISICDataset(train_imgs, train_msks, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ISICDataset(val_imgs,   val_msks,   val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ISICDataset(test_imgs,  test_msks,  val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'OK Dataset : {len(all_images)} images')
print(f'   Train:{len(train_imgs)} | Val:{len(val_imgs)} | Test:{len(test_imgs)}')

## BUILDING_CONSTRUCTION Étape 4 — Définition de DeepLabV3

In [ ]:
class DeepLabV3Wrapper(nn.Module):
    """ResNet50 pré-entraîné + ASPP, tête remplacée pour segmentation binaire."""
    def __init__(self, out_channels=1, pretrained=True):
        super().__init__()
        weights = seg_models.DeepLabV3_ResNet50_Weights.DEFAULT if pretrained else None
        self.model = seg_models.deeplabv3_resnet50(weights=weights, aux_loss=False)
        self.model.classifier[4] = nn.Conv2d(256, out_channels, kernel_size=1)
    def forward(self, x):
        out = self.model(x)['out']
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return out

# ── Instanciation ─────────────────────────────────────
deeplab = DeepLabV3Wrapper(pretrained=True).to(device)

print(f'GREEN_CIRCLE DeepLabV3 : {sum(p.numel() for p in deeplab.parameters() if p.requires_grad):>12,} paramètres')

# Test forward
with torch.no_grad():
    d = torch.randn(2,3,IMG_SIZE,IMG_SIZE).to(device)
    print(f'OK DeepLabV3 output : {deeplab(d).shape}')

## TOOL Étape 5 — Fonctions communes

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(weight, reduction='mean' if size_average else 'sum')
    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2.*intersection + 1) / (inputs.sum() + targets.sum() + 1)
        return bce_loss + dice_loss

def dice_coeff(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return (2. * intersection + 1) / (pred.sum() + target.sum() + 1)

def iou_coeff(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    pred = pred.view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1) / (union + 1)

def save_checkpoint(model, optimizer, epoch, loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)

def load_checkpoint(model, optimizer, path):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch'], checkpoint['loss']

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, device, save_path, patience=10):
    best_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': [], 'train_iou': [], 'val_iou': []}

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_dice = 0.0
        train_iou = 0.0

        for images, masks in tqdm(train_loader, desc=f'Époque {epoch+1}/{num_epochs}'):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_dice += dice_coeff(outputs, masks).item()
            train_iou += iou_coeff(outputs, masks).item()

        train_loss /= len(train_loader)
        train_dice /= len(train_loader)
        train_iou /= len(train_loader)

        model.eval()
        val_loss = 0.0
        val_dice = 0.0
        val_iou = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
                val_dice += dice_coeff(outputs, masks).item()
                val_iou += iou_coeff(outputs, masks).item()

        val_loss /= len(val_loader)
        val_dice /= len(val_loader)
        val_iou /= len(val_loader)

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dice)
        history['val_dice'].append(val_dice)
        history['train_iou'].append(train_iou)
        history['val_iou'].append(val_iou)

        print(f'Époque {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Dice: {train_dice:.4f} | Val Dice: {val_dice:.4f} | Train IoU: {train_iou:.4f} | Val IoU: {val_iou:.4f}')

        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0
            save_checkpoint(model, optimizer, epoch, val_loss, save_path)
            print(f'  → Meilleur modèle sauvegardé à {save_path}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Arrêt précoce à l\'époque {epoch+1}')
                break

    return history

## LAUNCH Étape 6 — Entraînement de DeepLabV3

In [ ]:
# ── Configuration ────────────────────────────────────
criterion = DiceBCELoss()
optimizer = optim.Adam(deeplab.parameters(), lr=5e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# Créer le dossier de sauvegarde
os.makedirs('outputs/deeplabv3', exist_ok=True)
save_path = 'outputs/deeplabv3/best_deeplabv3_isic.pth'

# ── Entraînement ──────────────────────────────────────
print('LAUNCH Démarrage de l\'entraînement de DeepLabV3...')
history = train_model(
    model=deeplab,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=50,
    device=device,
    save_path=save_path,
    patience=10
)

# Sauvegarder l'historique
with open('outputs/deeplabv3/history.json', 'w') as f:
    json.dump(history, f, indent=4)

print('OK Entraînement terminé !')

## REPORT Étape 7 — Évaluation de DeepLabV3

In [ ]:
# ── Charger le meilleur modèle ──────────────────────
checkpoint = torch.load(save_path)
deeplab.load_state_dict(checkpoint['model_state_dict'])
deeplab.eval()

# ── Évaluation sur le jeu de test ────────────────────
test_dice = 0.0
test_iou = 0.0
all_preds = []
all_targets = []

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Évaluation sur le test'):
        images, masks = images.to(device), masks.to(device)
        outputs = deeplab(images)
        test_dice += dice_coeff(outputs, masks).item()
        test_iou += iou_coeff(outputs, masks).item()
        all_preds.append(torch.sigmoid(outputs).cpu())
        all_targets.append(masks.cpu())

test_dice /= len(test_loader)
test_iou /= len(test_loader)

print(f'REPORT DeepLabV3 - Test Dice: {test_dice:.4f} | Test IoU: {test_iou:.4f}')

# ── Sauvegarder les métriques ────────────────────────
test_metrics = {
    'model': 'DeepLabV3',
    'test_dice': test_dice,
    'test_iou': test_iou,
    'history': history
}

with open('outputs/deeplabv3/test_metrics.json', 'w') as f:
    json.dump(test_metrics, f, indent=4)

# ── Tracer les courbes ───────────────────────────────
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history['train_dice'], label='Train Dice')
plt.plot(history['val_dice'], label='Val Dice')
plt.title('Dice Coefficient')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history['train_iou'], label='Train IoU')
plt.plot(history['val_iou'], label='Val IoU')
plt.title('IoU')
plt.legend()

plt.tight_layout()
plt.savefig('outputs/deeplabv3/training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print('OK Évaluation terminée !')

## TARGET Conclusion — DeepLabV3

**Résumé de l'entraînement :**
- **Modèle :** DeepLabV3 avec ResNet50 pré-entraîné
- **Paramètres :** {sum(p.numel() for p in deeplab.parameters() if p.requires_grad):,} entraînables
- **Époques :** 50 (avec arrêt précoce)
- **Learning Rate :** 5e-5 (avec scheduler ReduceLROnPlateau)
- **Loss :** Dice + BCE combinée
- **Métriques finales :**
  - Test Dice : {test_dice:.4f}
  - Test IoU : {test_iou:.4f}

**Points clés :**
- DeepLabV3 utilise l'architecture ASPP pour capturer le contexte multi-échelle
- Pré-entraînement sur ImageNet accélère la convergence
- Performance généralement supérieure aux modèles custom comme U-Net pour ce type de tâche

**Fichiers sauvegardés :**
- `outputs/deeplabv3/best_deeplabv3_isic.pth` : Meilleur modèle
- `outputs/deeplabv3/history.json` : Historique d'entraînement
- `outputs/deeplabv3/test_metrics.json` : Métriques de test
- `outputs/deeplabv3/training_curves.png` : Courbes d'entraînement

**Prochaines étapes :**
- Comparer avec U-Net et SegNet dans le notebook de comparaison
- Ajuster les hyperparamètres si nécessaire
- Tester sur de nouvelles images